# Dedicated Per-Drug MLP — Class Weights (+ LR from 07-01)

**Loads pre-saved aggregated data + LR results from 07-01.** Only runs MLP with class-weighted `CrossEntropyLoss`.

| Model | Details |
|---|---|
| **MLP** | 8x8 grid lr x dropout, class weights, external val scoring, patience=10 |

**Data:** Same species-stratified 70/15/15 split as 07-01 (loaded from saved indices + preprocessing states).
**Output:** Comparison with 07-01 LR + 07-01 MLP (no class weights) + 07-02 MLP (class weights).

Compatible with Google Colab.

In [ ]:
!pip install maldideepkit maldiamrkit seaborn --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from sklearn.metrics import (balanced_accuracy_score, roc_auc_score)

from maldideepkit.attention.mlp import SpectralAttentionMLP
from maldideepkit.base.data import fit_input_transform, apply_input_transform

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE_NAME = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE_NAME}")

In [ ]:
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

OUT_DIR = Path("./results_dedicated_lr_mlp_cw")
OUT_DIR.mkdir(exist_ok=True)

# Load from 07-01's saved state
SAVE_DIR = Path("./results_dedicated_lr_mlp")
assert SAVE_DIR.exists(), "Run 07-01 with the save cell first!"

DRUGS_10 = ["Ciprofloxacin", "Gentamicin", "Amoxicillin-Clavulanic acid",
            "Piperacillin-Tazobactam", "Cefepime", "Ceftriaxone",
            "Imipenem", "Ceftazidime", "Vancomycin", "Amikacin"]

LR_GRID = np.linspace(7.5e-5, 9.5e-5, 8)
DROP_GRID = np.linspace(0.5, 0.8, 8)
THRESHOLDS = np.linspace(0.05, 0.95, 91)

print(f"MLP grid: {len(LR_GRID)}x{len(DROP_GRID)}={len(LR_GRID)*len(DROP_GRID)}")

In [ ]:
# ── 1. LOAD PRE-SAVED DATA FROM 07-01 ──

loaded = {}  # {drug: {X_train_pp, y_train, X_val_pp, y_val, X_test_pp, y_test}}

for drug in DRUGS_10:
    safe_name = drug.replace(" ", "_").replace("-", "_")
    p = SAVE_DIR / f"data_{safe_name}.pkl"
    if not p.exists():
        print(f"  SKIP {drug}: no saved data at {p}")
        continue
    with open(p, "rb") as f:
        loaded[drug] = pickle.load(f)
    n_tr = len(loaded[drug]["y_train"])
    n_te = len(loaded[drug]["y_test"])
    print(f"  {drug:35s}  train={n_tr}  test={n_te}")

# Load LR results
lr_path = SAVE_DIR / "lr_results.pkl"
if lr_path.exists():
    with open(lr_path, "rb") as f:
        lr_results_saved = pickle.load(f)
    print(f"\nLoaded LR results for {len(lr_results_saved)} drugs")
else:
    lr_results_saved = {}
    print("\nNo LR results found -- run 07-01 first")

# Load 07-01 MLP results for comparison
mlp_path_07_01 = SAVE_DIR / "mlp_results.pkl"
if mlp_path_07_01.exists():
    with open(mlp_path_07_01, "rb") as f:
        mlp_07_01 = pickle.load(f)
    print(f"Loaded 07-01 MLP results for {len(mlp_07_01)} drugs")
else:
    mlp_07_01 = {}
    print("No 07-01 MLP results found")

In [ ]:
# ── 2. PYTORCH HELPERS ──

class BinDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

def build_mlp(dropout_high=0.4):
    return SpectralAttentionMLP(
        input_dim=6000, n_classes=2,
        hidden_dim=512, head_dims=(256, 128),
        dropout_high=dropout_high, dropout_low=dropout_high / 2.0,
        use_attention=False)

def predict_proba(model, X_np):
    model.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32).to(next(model.parameters()).device)
    with torch.no_grad():
        return torch.softmax(model(X_t), dim=1).cpu().numpy()[:, 1]

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 3. MLP WITH CLASS WEIGHTS
# ═══════════════════════════════════════════════════════════════════════════

def train_eval_mlp_cw(X_train, y_train, test_sets):
    """8x8 grid lr x dropout WITH class weights. Scored on external val set."""

    # Class weights computed from training labels
    from sklearn.utils.class_weight import compute_class_weight
    cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
    class_weights = torch.tensor(cw, dtype=torch.float32).to(DEVICE_NAME)

    # Internal val for early stopping (10% of train)
    from sklearn.model_selection import train_test_split
    X_ft, X_fv, y_ft, y_fv = train_test_split(
        X_train, y_train, test_size=0.1, stratify=y_train, random_state=SEED)

    best_ba, best_lr, best_dh = -1.0, None, None
    best_combo_model = None

    for lr_val in LR_GRID:
        for d in DROP_GRID:
            dh, dl = d, d / 2.0
            model = build_mlp(dh).to(DEVICE_NAME)
            train_dl = DataLoader(BinDataset(X_ft, y_ft), batch_size=64, shuffle=True)
            val_dl   = DataLoader(BinDataset(X_fv, y_fv), batch_size=128, shuffle=False)
            opt = torch.optim.AdamW(model.parameters(), lr=lr_val, weight_decay=1e-3)
            crit = nn.CrossEntropyLoss(weight=class_weights)
            best_vl, best_sd, patience = float("inf"), None, 0
            for ep in range(50):
                model.train()
                for xb, yb in train_dl:
                    xb, yb = xb.to(DEVICE_NAME), yb.to(DEVICE_NAME)
                    opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
                model.eval(); vl = 0.0
                with torch.no_grad():
                    for xb, yb in val_dl:
                        xb, yb = xb.to(DEVICE_NAME), yb.to(DEVICE_NAME)
                        vl += crit(model(xb), yb).item()
                vl /= len(val_dl)
                if vl < best_vl:
                    best_vl = vl; best_sd = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    patience = 0
                else:
                    patience += 1
                    if patience >= 10: break
            if best_sd is not None:
                model.load_state_dict(best_sd)
            proba = predict_proba(model, test_sets["Val"][0])
            ba = balanced_accuracy_score(test_sets["Val"][1], proba >= 0.5)
            if ba > best_ba:
                best_ba = ba; best_lr = lr_val; best_dh = dh
                best_combo_model = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Retrain best on FULL train with warmup
    model_final = build_mlp(best_dh).to(DEVICE_NAME)
    if best_combo_model is not None:
        model_final.load_state_dict(best_combo_model)
    train_dl = DataLoader(BinDataset(X_train, y_train), batch_size=64, shuffle=True)
    opt = torch.optim.AdamW(model_final.parameters(), lr=best_lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=90, eta_min=1e-6)
    crit = nn.CrossEntropyLoss(weight=class_weights)
    best_vl, best_sd, patience = float("inf"), None, 0
    for ep in range(100):
        model_final.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE_NAME), yb.to(DEVICE_NAME)
            if ep < 10:
                for pg in opt.param_groups: pg["lr"] = best_lr * (ep + 1) / 10
            opt.zero_grad(); crit(model_final(xb), yb).backward(); opt.step()
        if ep >= 10: sched.step()
        if ep % 5 == 0:
            model_final.eval(); vl = 0.0
            with torch.no_grad():
                for xb, yb in train_dl:
                    xb, yb = xb.to(DEVICE_NAME), yb.to(DEVICE_NAME)
                    vl += crit(model_final(xb), yb).item()
            vl /= len(train_dl)
            if vl < best_vl:
                best_vl = vl; best_sd = {k: v.cpu().clone() for k, v in model_final.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= 3: break
    if best_sd is not None:
        model_final.load_state_dict(best_sd)

    # Per-drug threshold tuned on external val set
    proba_val = predict_proba(model_final, test_sets["Val"][0])
    best_t = THRESHOLDS[np.argmax(
        [balanced_accuracy_score(test_sets["Val"][1], proba_val >= t) for t in THRESHOLDS])]

    results = {}
    for name, (X_te, y_te) in test_sets.items():
        proba = predict_proba(model_final, X_te)
        preds = proba >= best_t
        results[name] = {
            "BalAcc": balanced_accuracy_score(y_te, preds),
            "AUC": roc_auc_score(y_te, proba),
            "Threshold": best_t,
            "Best_Param": f"lr={best_lr:.1e} drop={best_dh:.1f}",
        }
    return results

---
## Per-Drug MLP (Class Weights)

In [ ]:
# ── 4. TRAIN MLP WITH CLASS WEIGHTS PER DRUG ──

mlp_cw_results = {}
for drug in tqdm(DRUGS_10, desc="MLP-CW"):
    if drug not in loaded:
        print(f"  SKIP {drug}: no loaded data")
        continue

    print(f"\n{'='*60}")
    print(f"  {drug}")
    print(f"{'='*60}")

    d = loaded[drug]
    test_sets = {"Val": (d["X_val_pp"], d["y_val"]), "Test": (d["X_test_pp"], d["y_test"])}

    print("  --- MLP (class weights) ---")
    mlp_cw_results[drug] = train_eval_mlp_cw(d["X_train_pp"], d["y_train"], test_sets)
    print(f"    Val BalAcc={mlp_cw_results[drug]['Val']['BalAcc']:.4f}  "
          f"Test BalAcc={mlp_cw_results[drug]['Test']['BalAcc']:.4f}")

print("\nMLP with class weights complete.")

---
## Results: LR + MLP (no CW) + MLP (CW)

In [ ]:
# ── 5. BUILD COMPARISON TABLE ──

rows = []
for drug in DRUGS_10:
    if drug not in mlp_cw_results: continue
    row = {"Drug": drug}

    if drug in lr_results_saved:
        row["LR"] = lr_results_saved[drug]["Test"]["BalAcc"]
    else:
        row["LR"] = np.nan

    if drug in mlp_07_01:
        row["MLP"] = mlp_07_01[drug]["Test"]["BalAcc"]
    else:
        row["MLP"] = np.nan

    row["MLP_CW"] = mlp_cw_results[drug]["Test"]["BalAcc"]
    rows.append(row)

df_comp = pd.DataFrame(rows).set_index("Drug")
short_names = {d: d[:15] for d in df_comp.index}

print("\nTest Balanced Accuracy:")
print(df_comp.round(4).to_string())

# Delta
if "MLP" in df_comp.columns and "MLP_CW" in df_comp.columns:
    print("\nDelta (MLP_CW - MLP):")
    for drug in df_comp.index:
        delta = df_comp.loc[drug, "MLP_CW"] - df_comp.loc[drug, "MLP"]
        print(f"  {drug:35s}  {delta:+.4f}")

In [ ]:
# ── Heatmap: LR vs MLP vs MLP_CW ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 3.5))

df_disp_ba = df_comp[["LR", "MLP", "MLP_CW"]]; df_disp_ba.index = [short_names[d] for d in df_disp_ba.index]
sns.heatmap(df_disp_ba.T, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0.5, vmax=1.0, linewidths=1.0, linecolor="white",
            cbar_kws={"label": "Balanced Accuracy", "shrink": 0.8}, ax=ax1)
ax1.set_title("Balanced Accuracy — LR / MLP / MLP+CW", fontsize=12, fontweight="bold")

# Delta heatmap
if "MLP" in df_comp.columns:
    delta_df = pd.DataFrame({d: [df_comp.loc[d, "MLP_CW"] - df_comp.loc[d, "MLP"]]
                              for d in df_comp.index}).rename(index={0: "Delta"})
    delta_df.columns = [short_names[d] for d in delta_df.columns]
    sns.heatmap(delta_df, annot=True, fmt="+.3f", cmap="RdBu_r", center=0,
                vmin=-0.05, vmax=0.05, linewidths=1.0, linecolor="white",
                cbar_kws={"label": "Delta (CW - no CW)", "shrink": 0.8}, ax=ax2)
    ax2.set_title("MLP+ClassWeight — MLP", fontsize=12, fontweight="bold")

fig.suptitle("Dedicated Per-Drug Models -- Aggregated 70/15/15 Test", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "heatmap_comparison.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Bar chart: LR vs MLP vs MLP_CW ──
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(df_comp)); w = 0.25
colors = {"LR": "#1f77b4", "MLP": "#ff7f0e", "MLP_CW": "#d62728"}
for i, (model, label) in enumerate(zip(["LR", "MLP", "MLP_CW"], ["LR", "MLP", "MLP+CW"])):
    vals = [df_comp.loc[d, model] for d in df_comp.index]
    ax.bar(x + (i - 1) * w, vals, w, label=label, color=colors[model], edgecolor="white", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([short_names[d] for d in df_comp.index], fontsize=8, rotation=45, ha="right")
ax.set_ylabel("Balanced Accuracy"); ax.set_title("LR vs MLP vs MLP+CW (Aggregated Test)")
ax.legend(fontsize=10); ax.set_ylim(0, 1)
ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
ax.grid(True, ls='--', lw=0.5, color='gray', alpha=0.5, axis='y')
plt.tight_layout()
plt.savefig(OUT_DIR / "barchart_comparison.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Save results ──
df_comp.to_csv(OUT_DIR / "comparison_lr_mlp_mlpcw.csv")

# Best params
param_rows = []
for drug in DRUGS_10:
    if drug not in mlp_cw_results: continue
    row = {"Drug": drug[:20], "MLP_CW": mlp_cw_results[drug]["Test"]["Best_Param"]}
    param_rows.append(row)
pd.DataFrame(param_rows).set_index("Drug").to_csv(OUT_DIR / "mlp_cw_best_params.csv")

print("\nSaved to", OUT_DIR.resolve())
for f in sorted(OUT_DIR.glob("*")):
    print(f"  {f.name}")

---
**Done.** Analysis complete.